# Module 2 Exercise: Hinting the Optimiser with `BROADCAST`

Catalyst picks a **Sort-Merge Join** by default when it cannot tell which side of
a join is small. If you know one side fits in memory, the `BROADCAST` hint
short-circuits the shuffle and ships the small table to every executor instead.

SQL Server analogue: this is the Spark equivalent of `INNER HASH JOIN`, but the
planner cost model is closer to a query hint like `OPTION (HASH JOIN)` than to
forcing a join type at parse time.

We will `EXPLAIN` the same join twice, once without the hint and once with it,
and compare the chosen physical operator.


## 1. Baseline plan, no hint

Join `orders` to `customers` on the foreign key. Without size stats to trust,
Catalyst falls back to a **SortMergeJoin**: both sides are shuffled and sorted
by the join key, then merged.


In [ ]:
%%sql
EXPLAIN
SELECT *
FROM orders o
JOIN customers c
  ON o.customer_id = c.id;


Look for `SortMergeJoin` in the physical plan, and `Exchange hashpartitioning(...)`
on both sides. Those two exchanges are a full shuffle, the expensive part.


## 2. Plan with `BROADCAST(c)`

Same query, one hint. Catalyst replaces the shuffle with a broadcast: the
`customers` side is collected to the driver and sent to every executor, so the
join becomes a **BroadcastHashJoin**.


In [ ]:
%%sql
EXPLAIN
SELECT /*+ BROADCAST(c) */ *
FROM orders o
JOIN  customers c
  ON o.customer_id = c.id;


Compare the two plans side by side:

- `SortMergeJoin` becomes `BroadcastHashJoin`
- Two `Exchange hashpartitioning` nodes collapse to one `BroadcastExchange`
- The large side (`orders`) is no longer shuffled

## Key takeaway

A broadcast join is the Spark equivalent of a nested-loop or hash join against a
small build-side table. Use it when one side is small (rule of thumb: < ~100 MB)
and the other is large. Misuse will OOM the driver, same risk as broadcasting a
huge lookup table in any distributed engine.
